# Resource Management Analysis: Data Extraction and Discovery

This notebook serves as the analytical engine to uncover the underlying resource dynamics of our event log. It is designed to provoke critical evaluation of the raw data, strip away assumptions, and extract the clear, factual guidelines needed to build a data-driven business process simulation model.


## Phase 1: Data Ingestion and Filtering

The first critical step is transforming the raw historical XES event log into a workable format. This phase ensures that the downstream simulation components are built on high-fidelity data.

* **Load XES Data:** Ingest the event log using `pm4py`.
* **Isolate Complete Traces:** Filter out running cases or those lacking a defined end state to prevent skewed temporal analysis.
* **Extract Resource Variables:** Clean the dataframe to isolate crucial columns: timestamps, activity names, and resource identifiers.

## Phase 2: Core Analytical Objectives

The following steps challenge the parsed data to extract the parameters required for the simulation's resource components. Remember that grading will be done on an individual basis, taking each person's workload into account.

### 1. Resource Availabilities (Section 1.6)

* **Basic Implementation:** Build a model that simulates resource availabilities based on an interval, e.g., a two-week interval.
* **Analytical Strategy:** Calculate the first and last timestamp for each resource per day to map working hours, shifts, and weekly active intervals.
* **Advanced Target:** Come up with an advanced model.



### 2. Resource Permissions (Section 1.7)

* **Basic Implementation:** Implement resource permissions based on whether a particular resource has processed an activity before.
* **Analytical Strategy:** Generate an Activity-Resource matrix to clearly map which resources are historically authorized to execute specific tasks.
* **Advanced Target:** Implement more advanced role-discovery approaches, e.g. [3].



### 3. Resource Allocation (Section 1.8)

* **Basic Implementation:** Implement only a random resource allocation.
* **Analytical Strategy:** Prepare the final parsed data structures that will feed the core engine, ensuring the random allocator has the correct pool of available and permitted resources to choose from.


## Evaluation & Outputs

For a very good grade, you (as an individual) will need to implement at least one advanced technique, explain and justify your design decisions, and provide empirical evaluations.

The final output of this notebook will be a clean, structured configuration file (e.g., `resource_profiles.json`) containing the distilled rules and parameters. This artifact will act as the foundational brain for the `ResourceManager` component within the central simulation engine.

In [1]:
# Core Data Manipulation
import pandas as pd
import numpy as np

# Process Mining Specifics
import pm4py

# Visualization (Crucial for identifying schedule intervals)
import matplotlib.pyplot as plt
import seaborn as sns

# Time & Date Handling
from datetime import datetime, timedelta

# Output Generation
import json

In [2]:
# Load the event log data
log_raw = pm4py.read_xes("../data/BPI Challenge 2017.xes.gz", variant= "r4pm")

"""
Attribute Renaming and filtering on complete events (lifeclycle:transition).
"""
def clean_data(log_raw):
    # Filter rows based on column: 'lifecycle:transition'
    log_raw = log_raw[log_raw['lifecycle:transition'] == "complete"]
    # Rename column 'MonthlyCost' to 'offer:MonthlyCost'
    log_raw = log_raw.rename(columns={'MonthlyCost': 'case:offer:MonthlyCost'})
    # Rename column 'OfferedAmount' to 'offer:OfferedAmount'
    log_raw = log_raw.rename(columns={'OfferedAmount': 'case:offer:OfferedAmount'})
    # Rename column 'Selected' to 'offer:Selected'
    log_raw = log_raw.rename(columns={'Selected': 'case:offer:Selected'})
    # Rename column 'Accepted' to 'offer:Accepted'
    log_raw = log_raw.rename(columns={'Accepted': 'case:offer:Accepted'})
    # Rename column 'CreditScore' to 'offer:CreditScore'
    log_raw = log_raw.rename(columns={'CreditScore': 'case:offer:CreditScore'})
    # Rename column 'FirstWithdrawalAmount' to 'offer:FirstWithdrawalAmount'
    log_raw = log_raw.rename(columns={'FirstWithdrawalAmount': 'case:offer:FirstWithdrawalAmount'})
    # Rename column 'NumberOfTerms' to 'offer:NumberOfTerms'
    log_raw = log_raw.rename(columns={'NumberOfTerms': 'case:offer:NumberOfTerms'})
    # Convert 'time:timestamp' to datetime dtype
    log_raw['time:timestamp'] = pd.to_datetime(log_raw['time:timestamp'], errors='coerce') # WHY ARE THERE ERRORS??
    return log_raw

log_all_cases = clean_data(log_raw.copy())

# Select only complete cases with following endpoint activities: A Cancelled, A Denied, A Pending, O Cancelled, or O Refused)
print("Number of cases before filtering for complete cases: ", log_all_cases['case:concept:name'].nunique())
log_all_complete_cases = pm4py.filter_end_activities(log_all_cases, ["A_Validating", "W_Complete application", "A_Complete"], retain= False)
print("Number of cases after filtering for complete cases: ", log_all_complete_cases['case:concept:name'].nunique())

Number of cases before filtering for complete cases:  31509
Number of cases after filtering for complete cases:  31413


In [5]:
# Build intervals of availability for each resource
# Resource, day, first timestamp, last timestamp, list_of_activities
def build_resource_intervals(log_all_complete_cases):
    df = log_all_complete_cases.copy()
    df['date'] = df['time:timestamp'].dt.normalize()

    resource_intervals = {}
    for resource, group in df.groupby('org:resource'):
        intervals = []
        for day, day_group in group.groupby('date'):
            first_timestamp = day_group['time:timestamp'].min()
            last_timestamp = day_group['time:timestamp'].max()
            activities = day_group['concept:name'].tolist()
            intervals.append((day, first_timestamp, last_timestamp, activities))
        resource_intervals[resource] = intervals
    return resource_intervals

resource_intervals = build_resource_intervals(log_all_complete_cases)

In [6]:
# Convert resource_intervals into JSON
resource_intervals_json = []
for resource, intervals in resource_intervals.items():
    for day, first_ts, last_ts, activities in intervals:
        resource_intervals_json.append({
            "date": day.strftime("%Y-%m-%d"),
            "resource": resource,
            "first_timestamp": first_ts.isoformat(),
            "last_timestamp": last_ts.isoformat(),
            "activities": activities,
        })

with open("resource_intervals.json", "w", encoding="utf-8") as f:
    json.dump(resource_intervals_json, f, indent=2)

resource_intervals_json[:5]

[{'date': '2016-01-01',
  'resource': 'User_1',
  'first_timestamp': '2016-01-01T09:51:15.304000+00:00',
  'last_timestamp': '2016-01-01T21:58:53.572000+00:00',
  'activities': ['A_Create Application',
   'A_Submitted',
   'A_Concept',
   'A_Create Application',
   'A_Submitted',
   'A_Concept',
   'A_Create Application',
   'A_Submitted',
   'A_Concept',
   'A_Create Application',
   'A_Submitted',
   'A_Create Application',
   'A_Submitted',
   'A_Concept',
   'A_Create Application',
   'A_Submitted',
   'A_Concept',
   'A_Create Application',
   'A_Submitted',
   'A_Concept',
   'A_Create Application',
   'A_Submitted',
   'A_Concept',
   'A_Create Application',
   'A_Submitted',
   'A_Concept',
   'A_Create Application',
   'A_Submitted',
   'A_Concept',
   'A_Create Application',
   'A_Submitted',
   'A_Concept',
   'A_Create Application',
   'A_Submitted',
   'A_Concept',
   'A_Create Application',
   'A_Submitted',
   'A_Concept',
   'A_Create Application',
   'A_Submitted',
   

In [7]:
# Identify gaps in resource availability (weekends, nights, vacation periods)

def classify_gap(days):
    weekend_days = sum(1 for d in days if d.weekday() >= 5)
    weekday_days = len(days) - weekend_days
    if weekday_days >= 2 and weekend_days == 0:
        return "vacation"
    if weekend_days == len(days) and len(days) <= 2:
        return "weekend"
    if weekday_days > 0 and weekend_days > 0:
        return "mixed"
    return "weekday_gap"

resource_gaps = []
resource_summaries = []
for resource, intervals in resource_intervals.items():
    active_days = sorted(day for day, _, _, _ in intervals)
    if not active_days:
        continue

    full_span = pd.date_range(active_days[0], active_days[-1], freq="D")
    active_set = set(active_days)
    missing_days = [day for day in full_span if day not in active_set]

    gap_blocks = []
    current_block = []
    for day in missing_days:
        if not current_block or day == current_block[-1] + pd.Timedelta(days=1):
            current_block.append(day)
        else:
            gap_blocks.append(current_block)
            current_block = [day]
    if current_block:
        gap_blocks.append(current_block)

    for block in gap_blocks:
        gap_type = classify_gap(block)
        resource_gaps.append({
            "resource": resource,
            "gap_start": block[0].strftime("%Y-%m-%d"),
            "gap_end": block[-1].strftime("%Y-%m-%d"),
            "gap_length_days": len(block),
            "gap_type": gap_type,
            "weekend_days": sum(1 for d in block if d.weekday() >= 5),
            "weekday_days": sum(1 for d in block if d.weekday() < 5),
        })

    start_hours = [first_ts.hour + first_ts.minute / 60 for _, first_ts, _, _ in intervals]
    end_hours = [last_ts.hour + last_ts.minute / 60 for _, _, last_ts, _ in intervals]
    resource_summaries.append({
        "resource": resource,
        "active_days": len(active_days),
        "avg_start_hour": np.mean(start_hours) if start_hours else None,
        "avg_end_hour": np.mean(end_hours) if end_hours else None,
        "avg_work_duration_hours": np.mean([
            (last_ts - first_ts).total_seconds() / 3600 for _, first_ts, last_ts, _ in intervals
        ]) if intervals else None,
    })

resource_gaps_df = pd.DataFrame(resource_gaps).sort_values(["gap_length_days", "resource"], ascending=[False, True])
resource_summary_df = pd.DataFrame(resource_summaries).sort_values("resource")

resource_gaps_df.head(10), resource_summary_df.head(10)

(      resource   gap_start     gap_end  gap_length_days gap_type  \
 4857   User_82  2016-03-03  2016-08-21              172    mixed   
 1428  User_142  2016-01-31  2016-07-06              158    mixed   
 1430  User_142  2016-07-11  2016-12-15              158    mixed   
 228   User_103  2016-06-04  2016-09-26              115    mixed   
 1923   User_21  2016-02-26  2016-06-12              108    mixed   
 4827    User_8  2016-02-23  2016-06-08              107    mixed   
 2131   User_25  2016-08-31  2016-12-14              106    mixed   
 2034   User_23  2016-11-01  2017-01-31               92    mixed   
 358    User_11  2016-04-10  2016-07-03               85    mixed   
 1091  User_128  2016-08-04  2016-10-23               81    mixed   
 
       weekend_days  weekday_days  
 4857            50           122  
 1428            45           113  
 1430            44           114  
 228             34            81  
 1923            32            76  
 4827            30    

In [8]:
# Build a resource availability model for future simulation

def build_resource_availability_model(resource_intervals):
    model = []
    permissions_by_resource = {}

    for resource, intervals in resource_intervals.items():
        all_activities = sorted({
            activity
            for _, _, _, activity_list in intervals
            for activity in activity_list
        })
        permissions_by_resource[resource] = all_activities

        for day, first_ts, last_ts, activities in sorted(intervals, key=lambda x: x[0]):
            day_start = pd.Timestamp(day)
            day_end = day_start + pd.Timedelta(days=1)
            available_windows = [
                {
                    "start": first_ts.isoformat(),
                    "end": last_ts.isoformat(),
                    "activities": activities,
                }
            ]

            unavailable_windows = []
            if first_ts > day_start:
                unavailable_windows.append({
                    "start": day_start.isoformat(),
                    "end": first_ts.isoformat(),
                    "reason": "not available before first shift",
                })
            if last_ts < day_end:
                unavailable_windows.append({
                    "start": last_ts.isoformat(),
                    "end": day_end.isoformat(),
                    "reason": "not available after last shift",
                })

            model.append({
                "resource": resource,
                "date": day.strftime("%Y-%m-%d"),
                "available": True,
                "available_windows": available_windows,
                "unavailable_windows": unavailable_windows,
                "permissions": all_activities,
            })

    return model, permissions_by_resource

resource_availability_model, resource_permissions = build_resource_availability_model(resource_intervals)

with open("resource_availability_model.json", "w", encoding="utf-8") as f:
    json.dump(resource_availability_model, f, indent=2)

resource_availability_model[:5]

[{'resource': 'User_1',
  'date': '2016-01-01',
  'available': True,
  'available_windows': [{'start': '2016-01-01T09:51:15.304000+00:00',
    'end': '2016-01-01T21:58:53.572000+00:00',
    'activities': ['A_Create Application',
     'A_Submitted',
     'A_Concept',
     'A_Create Application',
     'A_Submitted',
     'A_Concept',
     'A_Create Application',
     'A_Submitted',
     'A_Concept',
     'A_Create Application',
     'A_Submitted',
     'A_Create Application',
     'A_Submitted',
     'A_Concept',
     'A_Create Application',
     'A_Submitted',
     'A_Concept',
     'A_Create Application',
     'A_Submitted',
     'A_Concept',
     'A_Create Application',
     'A_Submitted',
     'A_Concept',
     'A_Create Application',
     'A_Submitted',
     'A_Concept',
     'A_Create Application',
     'A_Submitted',
     'A_Concept',
     'A_Create Application',
     'A_Submitted',
     'A_Concept',
     'A_Create Application',
     'A_Submitted',
     'A_Concept',
     'A_Create